In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from utils import *

In [ ]:
df= pd.read_parquet('training_data/estimator_training_data.parquet')
FEATURES = df.select_dtypes(include=[np.number]).columns.tolist()
PRIMARY_KEY = "RowKey"


In [ ]:
from utils import PlotCorrelationMatrix

# Select numeric columns only
df_num = df.select_dtypes(include="number")

# Plot correlation matrix
pcm = PlotCorrelationMatrix(df_num, FEATURES, (14, 12))


In [ ]:
# Let us train a regression model to estimate sparsity and shannon_entropy.
from sklearn.model_selection import train_test_split

TARGET_SPARSITY = "statevector_saved_sparsity"
TARGET_SHANNON_ENTROPY = "statevector_saved_shannon_entropy"
# TARGET_AGG_VN_ENTROPY = "vn_entropy_std_log"
FEATURES_CLEAN = [
    f for f in FEATURES
    if f not in [TARGET_SHANNON_ENTROPY, TARGET_SPARSITY, PRIMARY_KEY]
]
X = df[FEATURES_CLEAN]
y_sparsity = df[TARGET_SPARSITY]
y_shannon_entropy = df[TARGET_SHANNON_ENTROPY]

X_train, X_test, y_s_train, y_s_test, y_e_train, y_e_test = train_test_split(
    X, y_sparsity, y_shannon_entropy, 
    test_size=0.2,
    random_state=42
)
print(len(X), len(X_train), len(X_test))
print(X_train.columns.tolist())

In [ ]:
from sklearn.ensemble import RandomForestRegressor

sparsity_model = RandomForestRegressor(
    n_estimators=100,      # ↓ from 300
    max_depth=15,          # LIMIT depth
    min_samples_leaf=5,    # prevent overfitting
    random_state=42,
    n_jobs=-1
)

shannon_entropy_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)


sparsity_model.fit(X_train, y_s_train)
shannon_entropy_model.fit(X_train, y_e_train)


In [ ]:
from sklearn.linear_model import LinearRegression

# Linear regression models
sparsity_lr = LinearRegression(n_jobs=-1)
shannon_entropy_lr = LinearRegression(n_jobs=-1)

# Fit models
sparsity_lr.fit(X_train, y_s_train)
shannon_entropy_lr.fit(X_train, y_e_train)


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score


def evaluate(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    print(f"{name}")
    print(f"  R²   : {r2_score(y_test, y_pred):.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print()

evaluate(sparsity_model, X_test, y_s_test, "Sparsity Model")
evaluate(shannon_entropy_model, X_test, y_e_test, "Shannon Entropy Model")

evaluate(sparsity_lr, X_test, y_s_test, "Sparsity Model (Linear Regression)")
evaluate(shannon_entropy_lr, X_test, y_e_test, "Shannon Entropy Model (Linear Regression)")


In [ ]:
plotn = 1000
new_X = X_test.iloc[:plotn]

sparsity_pred = sparsity_model.predict(new_X)
shannon_entropy_pred = shannon_entropy_model.predict(new_X)

actual_sparsity = y_s_test.iloc[:plotn]
shannon_actual_entropy = y_e_test.iloc[:plotn]


fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# --- Sparsity ---
axes[0].scatter(actual_sparsity, sparsity_pred)
axes[0].plot(
    [actual_sparsity.min(), actual_sparsity.max()],
    [actual_sparsity.min(), actual_sparsity.max()]
)
axes[0].set_xlabel("Actual Sparsity", fontsize=26)
axes[0].set_ylabel("Predicted Sparsity", fontsize=26)
axes[0].set_title("Sparsity: Predicted vs Actual", fontsize=26)

# --- Shannon Entropy ---
axes[1].scatter(shannon_actual_entropy, shannon_entropy_pred)
axes[1].plot(
    [shannon_actual_entropy.min(), shannon_actual_entropy.max()],
    [shannon_actual_entropy.min(), shannon_actual_entropy.max()]
)
axes[1].set_xlabel("Actual Entropy", fontsize=26)
axes[1].set_ylabel("Predicted Entropy", fontsize=26)
axes[1].set_title("Shannon Entropy: Predicted vs Actual", fontsize=26)

# Tick labels
for ax in axes:
    ax.tick_params(axis='both', labelsize=16)

plt.tight_layout()
plt.show()


# Sparsity Bin Classifier

In [ ]:
bins = [
    0.0,
    0.1,
    0.2,
    0.4,  
    0.6,  
    1.01  
]

# bins = [0.0, 0.5, 0.6, 1.01]

labels = [str(bins[i])+ "-" + str(bins[i+1]) for i in range(len(bins)-1)]

df["sparsity_bin"] = pd.cut(
    df["statevector_saved_sparsity"],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURES_CLEAN]
y = df["sparsity_bin"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelSparsityClassifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=200,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

modelSparsityClassifier.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_pred = modelSparsityClassifier.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred,
        digits=3
    )
)

In [ ]:
bin_counts = df["sparsity_bin"].value_counts().sort_index()
total = bin_counts.sum()

print("Sparsity bin counts:")
for label, count in bin_counts.items():
    pct = 100 * count / total
    print(f"{label:10s}: {count:8d}  ({pct:6.2f}%)")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

labels = modelSparsityClassifier.classes_

cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    annot_kws={"size": 14}   # numbers inside cells
)

plt.xlabel("Predicted sparsity bin", fontsize=16)
plt.ylabel("True sparsity bin", fontsize=16)
plt.title("Confusion Matrix (Counts)", fontsize=16)

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

labels = modelSparsityClassifier.classes_

cm_norm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels,
    normalize="true"
)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    annot_kws={"size": 14}   # numbers inside cells
)

plt.xlabel("Predicted sparsity bin", fontsize=16)
plt.ylabel("True sparsity bin", fontsize=16)
plt.title("Confusion Matrix (Row-normalized)", fontsize=16)

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
importance = pd.Series(
    modelSparsityClassifier.feature_importances_,
    index=FEATURES_CLEAN
).sort_values(ascending=False)

importance.head(20)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# Plot Distribution of Statevector_saved_sparsity
sns.histplot(df[TARGET_SPARSITY], kde=True, ax=axes[0])
axes[0].set_title('Distribution of Statevector_saved_sparsity', fontsize=26)
axes[0].set_xlabel('Statevector_saved_sparsity',fontsize=26)
axes[0].set_ylabel('Frequency',fontsize=26)
axes[0].grid(True)

# Plot Distribution of Statevector_saved_entropy
sns.histplot(df[TARGET_SHANNON_ENTROPY], kde=True, ax=axes[1])
axes[1].set_title('Distribution of Statevector_saved_entropy',fontsize=26)
axes[1].set_xlabel('Statevector_saved_entropy',fontsize=26)
axes[1].set_ylabel('Frequency',fontsize=26)
axes[1].grid(True)
for ax in axes:
    ax.tick_params(axis='both', labelsize=20)
plt.tight_layout()
plt.show()